In [1]:
!pip install neuralforecast --quiet
!pip install statsforecast  --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 62.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.5 which is incompatible.
     ━━━━━━

In [3]:
import time
import pandas as pd
import numpy as np
#from sklearn.preprocessing import LabelEncoder
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SimpleExponentialSmoothingOptimized
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS, NHITS, PatchTST
from neuralforecast.auto import AutoNHITS, AutoPatchTST, AutoTimesNet
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import smape, mase, rmsse
from functools import partial
import warnings
warnings.filterwarnings('ignore')




In [4]:
# ----------------------------------------------------------------------
# 1. Загрузка и подготовка данных (как в исходном ноутбуке)
# ----------------------------------------------------------------------
def load_and_preprocess(file_path):
    df = pd.read_excel(file_path)
    df = df.drop_duplicates()
    df["unique_id"] = df["id"].astype(str) + "_" + df["Код филиала"].astype(str) + "_" + df["Код ЦФО"].astype(str)
    df_both = df[df["Показатель"].isin(["ДЗ н.п.", "ДЗ к.п."])]
    id_vars = ["id", "Код филиала", "Код ЦФО", "Показатель", "number", "unique_id"]
    df_long = pd.melt(df_both, id_vars=id_vars, var_name="month", value_name="y")
    def get_date(row):
        month_num = int(row['month'])
        if month_num <= 12:
            year, month = 2022, month_num
        else:
            year, month = 2023, month_num - 12
        if row['Показатель'] == 'ДЗ н.п.':
            return pd.Timestamp(f"{year}-{month:02d}-01")
        else:
            if month == 12:
                next_year, next_month = year + 1, 1
            else:
                next_year, next_month = year, month + 1
            next_month_start = pd.Timestamp(f"{next_year}-{next_month:02d}-01")
            return next_month_start - pd.Timedelta(days=1)
    df_long['ds'] = df_long.apply(get_date, axis=1)
    mask = df_long.groupby("unique_id")["y"].transform("sum") != 0
    df_long = df_long[mask].sort_values(["unique_id", "ds"])
    df_start = df_long[df_long['Показатель'] == 'ДЗ н.п.'].copy()
    df_end = df_long[df_long['Показатель'] == 'ДЗ к.п.'].copy()
    df_end['ds'] = df_end['ds'] + pd.offsets.MonthBegin(1)
    df_combined = pd.concat([df_start, df_end], ignore_index=True)
    df_combined = df_combined.sort_values(['unique_id', 'ds']).drop_duplicates(subset=['unique_id', 'ds'], keep='first')
    return df_combined[['unique_id', 'y', 'ds']]

def classify_time_series(df_long):
    results = []
    for uid, group in df_long.groupby('unique_id'):
        T = len(group)
        non_zero_values = group[group['y'] > 0]['y']
        N = len(non_zero_values)
        adi = T / N if N > 0 else np.inf
        if N == 0:
            cv2 = np.nan
        elif N == 1:
            cv2 = 0.0
        else:
            mean = non_zero_values.mean()
            std = non_zero_values.std(ddof=1)
            cv = std / mean if mean != 0 else np.inf
            cv2 = cv ** 2
        adi_threshold, cv2_threshold = 1.32, 0.49
        if adi <= adi_threshold and cv2 <= cv2_threshold:
            demand_class = "Smooth"
        elif adi <= adi_threshold and cv2 > cv2_threshold:
            demand_class = "Erratic"
        elif adi > adi_threshold and cv2 <= cv2_threshold:
            demand_class = "Intermittent"
        elif adi > adi_threshold and cv2 > cv2_threshold:
            demand_class = "Lumpy"
        else:
            demand_class = "Undefined"
        results.append({'unique_id': uid, 'adi': adi, 'cv2': cv2, 'class': demand_class})
    return pd.DataFrame(results)

def split_train_test(df_long, split_date):
    train = df_long[df_long['ds'] <= split_date].copy()
    test = df_long[df_long['ds'] > split_date].copy()
    return train, test

def filter_by_class(df, classification_df, class_name):
    ids = classification_df[classification_df['class'] == class_name]['unique_id'].values
    return df[df['unique_id'].isin(ids)].copy()

HORIZON = 4
split_date = pd.to_datetime("2023-09-01")

# Загружаем данные
df_long = load_and_preprocess("data.xlsx")
classification_df = classify_time_series(df_long)
train_all, test_all = split_train_test(df_long, split_date)

In [5]:
df_long.head()

,unique_id,y,ds
0,10_1000_14,2.597909e+08,2022-01-01
1,10_1000_14,2.481821e+08,2022-02-01
2,10_1000_14,2.740626e+08,2022-03-01
3,10_1000_14,3.083320e+08,2022-04-01
4,10_1000_14,2.889103e+08,2022-05-01


In [6]:
classification_df['class'].value_counts()

,count
class,
Smooth,576
Intermittent,513
Erratic,188
Lumpy,145
Undefined,9


In [7]:
# ----------------------------------------------------------------------
# 2. Базовая модель AutoARIMA (обучение на всех рядах)
# ----------------------------------------------------------------------
def train_autoarima(train_df, test_df, model_name="AutoARIMA"):
    start = time.time()
    sf = StatsForecast(models=[AutoARIMA()], freq='MS', n_jobs=-1)
    forecast = sf.forecast(df=train_df, h=HORIZON)
    forecast = test_df.merge(forecast, on=['unique_id','ds'], how='left')
    train_time = time.time() - start
    return forecast, train_time

baseline_forecast, baseline_time = train_autoarima(train_all, test_all)
print(f"Baseline AutoARIMA обучена за {baseline_time:.2f} сек")

Baseline AutoARIMA обучена за 359.89 сек


In [8]:
# ----------------------------------------------------------------------
# 3. Функция оценки метрик
# ----------------------------------------------------------------------
def evaluate_forecast(forecast_df, train_df, model_col):
    eval_df = evaluate(
        forecast_df[['unique_id','ds','y',model_col]],
        metrics=[smape, partial(mase, seasonality=12), partial(rmsse, seasonality=12)],
        train_df=train_df,
        models=[model_col]
    )
    median_metrics = eval_df.groupby('metric')[model_col].median().to_dict()
    return median_metrics

In [9]:
# ----------------------------------------------------------------------
# 4. Реализация сценариев
# ----------------------------------------------------------------------
# Сценарий 1: двухсегментная схема
# Группа A: Smooth + Erratic -> SimpleExponentialSmoothingOptimized
# Группа B: Intermittent + Lumpy -> AutoARIMA
def scenario1(train_all, test_all, classification_df):
    groupA_ids = classification_df[classification_df['class'].isin(['Smooth','Erratic'])]['unique_id'].values
    groupB_ids = classification_df[classification_df['class'].isin(['Intermittent','Lumpy'])]['unique_id'].values

    trainA = train_all[train_all['unique_id'].isin(groupA_ids)]
    testA = test_all[test_all['unique_id'].isin(groupA_ids)]
    trainB = train_all[train_all['unique_id'].isin(groupB_ids)]
    testB = test_all[test_all['unique_id'].isin(groupB_ids)]

    # Модель для группы A
    startA = time.time()
    modelA = SimpleExponentialSmoothingOptimized()
    sfA = StatsForecast(models=[modelA], freq='MS', n_jobs=-1)
    forecastA = sfA.forecast(df=trainA, h=HORIZON)
    forecastA = testA.merge(forecastA, on=['unique_id','ds'], how='left')
    timeA = time.time() - startA
    # Определяем имя колонки модели
    modelA_name = forecastA.columns[-1]  # последняя колонка после merge

    # Модель для группы B
    startB = time.time()
    modelB = AutoARIMA()
    sfB = StatsForecast(models=[modelB], freq='MS', n_jobs=-1)
    forecastB = sfB.forecast(df=trainB, h=HORIZON)
    forecastB = testB.merge(forecastB, on=['unique_id','ds'], how='left')
    timeB = time.time() - startB
    modelB_name = forecastB.columns[-1]

    # Объединяем
    forecast_all = pd.concat([forecastA, forecastB], ignore_index=True)
    # Создаём единую колонку 'forecast'
    forecast_all['forecast'] = forecast_all[modelA_name].fillna(forecast_all[modelB_name])
    forecast_all = forecast_all[['unique_id','ds','y','forecast']]
    return forecast_all, (timeA, timeB)

# Сценарий 2: полный конвейер (индивидуальные чемпионы)
def scenario2(train_all, test_all, classification_df):
    class_to_model = {
        'Smooth': NBEATS(h=HORIZON, input_size=16, scaler_type='robust', max_steps=500,
                         early_stop_patience_steps=-1, accelerator='auto', random_seed=42),
        'Erratic': AutoPatchTST(h=HORIZON),
        'Intermittent': AutoARIMA(),
        'Lumpy': AutoARIMA()
    }
    forecasts_list = []
    times = {}
    for class_name, model in class_to_model.items():
        train_class = filter_by_class(train_all, classification_df, class_name)
        test_class = filter_by_class(test_all, classification_df, class_name)
        if len(train_class['unique_id'].unique()) == 0:
            continue
        start = time.time()
        if isinstance(model, AutoARIMA):
            sf = StatsForecast(models=[model], freq='MS', n_jobs=-1)
            forecast = sf.forecast(df=train_class, h=HORIZON)
            forecast = test_class.merge(forecast, on=['unique_id','ds'], how='left')
            model_name = forecast.columns[-1]  # динамическое имя
        else:
            nf = NeuralForecast(models=[model], freq='MS')
            nf.fit(df=train_class)
            forecast = nf.predict()
            forecast = test_class.merge(forecast, on=['unique_id','ds'], how='left')
            model_name = forecast.columns[-1]  # динамическое имя
        times[class_name] = time.time() - start
        forecast['forecast'] = forecast[model_name]
        forecasts_list.append(forecast[['unique_id','ds','y','forecast']])
    forecast_all = pd.concat(forecasts_list, ignore_index=True)
    return forecast_all, times




In [10]:
# ----------------------------------------------------------------------
# 5. Запуск сценариев и сбор результатов
# ----------------------------------------------------------------------
print("\n=== Запуск сценариев ===")
# Baseline уже есть
baseline_metrics = evaluate_forecast(baseline_forecast, train_all, 'AutoARIMA')

# Сценарий 1
s1_forecast, s1_times = scenario1(train_all, test_all, classification_df)
s1_metrics = evaluate_forecast(s1_forecast, train_all, 'forecast')
# Время обучения: сумма времени для двух групп
s1_total_time = sum(s1_times)

# Сценарий 2
s2_forecast, s2_times = scenario2(train_all, test_all, classification_df)
s2_metrics = evaluate_forecast(s2_forecast, train_all, 'forecast')
s2_total_time = sum(s2_times.values())




=== Запуск сценариев ===


INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.4 M  | train
-------------------------------------------------------
2.4 M     Trainable params
180       Non-trainable params
2.4 M     Total params
9.639     Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=1000` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [11]:
# ----------------------------------------------------------------------
# 6. Вывод результатов
# ----------------------------------------------------------------------
print("\n" + "="*60)
print("СРАВНЕНИЕ СЦЕНАРИЕВ (медианные метрики)")
print("="*60)
results_df = pd.DataFrame({
    'Baseline (AutoARIMA)': baseline_metrics,
    'Сценарий 1 (двухсегментный)': s1_metrics,
    'Сценарий 2 (полный конвейер)': s2_metrics
}).T
print(results_df.round(4))

print("\n" + "="*60)
print("ВРЕМЯ ОБУЧЕНИЯ")
print("="*60)
print(f"Baseline (AutoARIMA на всех рядах):        {baseline_time:.2f} сек")
print(f"Сценарий 1 (SESOptimized + AutoARIMA):     {s1_total_time:.2f} сек")
print(f"  - SESOptimized на Smooth+Erratic:        {s1_times[0]:.2f} сек")
print(f"  - AutoARIMA на Intermittent+Lumpy:       {s1_times[1]:.2f} сек")
print(f"Сценарий 2 (индивидуальные чемпионы):      {s2_total_time:.2f} сек")
for cls, t in s2_times.items():
    print(f"  - {cls}: {t:.2f} сек")


СРАВНЕНИЕ СЦЕНАРИЕВ (медианные метрики)
                                mase   rmsse   smape
Baseline (AutoARIMA)          0.2900  0.2647  0.1021
Сценарий 1 (двухсегментный)   0.2234  0.2280  0.0929
Сценарий 2 (полный конвейер)  0.1997  0.2022  0.0873

ВРЕМЯ ОБУЧЕНИЯ
Baseline (AutoARIMA на всех рядах):        359.89 сек
Сценарий 1 (SESOptimized + AutoARIMA):     173.65 сек
  - SESOptimized на Smooth+Erratic:        2.41 сек
  - AutoARIMA на Intermittent+Lumpy:       171.24 сек
Сценарий 2 (индивидуальные чемпионы):      1010.17 сек
  - Smooth: 12.42 сек
  - Erratic: 805.72 сек
  - Intermittent: 141.66 сек
  - Lumpy: 50.37 сек


In [12]:
# ----------------------------------------------------------------------
# 7. Дополнительно: качество по классам для каждого сценария
# ----------------------------------------------------------------------
def metrics_by_class(forecast_df, classification_df, train_all):
    class_metrics = {}
    for class_name in classification_df['class'].unique():
        ids = classification_df[classification_df['class']==class_name]['unique_id'].values
        sub = forecast_df[forecast_df['unique_id'].isin(ids)]
        if len(sub) == 0:
            continue
        class_metrics[class_name] = evaluate_forecast(sub, train_all, 'forecast')
    return pd.DataFrame(class_metrics).T

print("\n" + "="*60)
print("МЕТРИКИ ПО КЛАССАМ ДЛЯ СЦЕНАРИЯ 2")
print("="*60)
metrics_class_s2 = metrics_by_class(s2_forecast, classification_df, train_all)
print(metrics_class_s2.round(4))

# Сохраняем результаты
results_df.to_csv('scenarios_comparison.csv')
print("\nРезультаты сохранены в 'scenarios_comparison.csv'")


МЕТРИКИ ПО КЛАССАМ ДЛЯ СЦЕНАРИЯ 2
                mase   rmsse   smape
Smooth        0.5391  0.5485  0.1015
Erratic       0.1447  0.1466  0.2041
Intermittent  0.0000  0.0000  0.0000
Lumpy         0.0465  0.0481  0.4752

Результаты сохранены в 'scenarios_comparison.csv'


In [13]:
print("\n" + "="*60)
print("МЕТРИКИ ПО КЛАССАМ ДЛЯ СЦЕНАРИЯ 1")
print("="*60)
metrics_class_s1 = metrics_by_class(s1_forecast, classification_df, train_all)
print(metrics_class_s1.round(4))


МЕТРИКИ ПО КЛАССАМ ДЛЯ СЦЕНАРИЯ 1
                mase   rmsse   smape
Smooth        0.5915  0.5719  0.1057
Erratic       0.2129  0.1834  0.2262
Intermittent  0.0000  0.0000  0.0000
Lumpy         0.0465  0.0481  0.4752


In [15]:
# Сохраняем результаты прогнозов
s1_forecast.to_csv('s1_forecast.csv')
s2_forecast.to_csv('s2_forecast.csv')
print("\nРезультаты сохранены")




Результаты сохранены
